In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

CATALOG = "workspace"
SILVER_SCHEMA = "ecommerce_silver"
GOLD_SCHEMA = "ecommerce_gold"

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}"

)

def read_silver(table_name: str) -> DataFrame:
    return spark.table(
        f"{CATALOG}.{SILVER_SCHEMA}.{table_name}"
    )

def save_gold(
    dataframe: DataFrame,
    table_name: str,

) -> None: 
    target_table = (
        f"{CATALOG}.{GOLD_SCHEMA}.{table_name}"
    )

    (
        dataframe.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", True)
        .saveAsTable(target_table)

    )

    print(
        f"Table saved: {target_table}"
        f"({dataframe.count():,} rows)"

    )

In [0]:
amazon_products = read_silver("amazon_products")
amazon_product_catalog = (
    amazon_products
    .select(
        "product_id",
        "product_name",
        "manufacturer",
        "price",
        "category",
        "dimensions",
        "about_product",
        "description",
        "_ingested_at",
        "_source_file_name",

    )
    .withColumn(
        "manufacturer",
        F.coalesce(
            F.col("manufacturer"),
            F.lit("Unknown"),
        )

    )
    .withColumn(
        "category",
        F.coalesce(
            F.col("category"),
            F.lit("Unknown"),
        )

    )
    .withColumn(
        "currency",
        F.lit("USD"),

    )
    .withColumn(
        "has_dimensions",
        F.col("dimensions").isNotNull()
        & (F.length(F.trim("dimensions")) > 10),

    )
    .withColumn(
        "has_description",
        F.col("description").isNotNull()
        & (F.length(F.trim("description")) > 0),

    )
    .withColumn(
        "has_about_product",
        F.col("about_product").isNotNull()
        & (F.length(F.trim("about_product")) > 0),
    )
    .withColumn(
        "data_completeness_score",
        (
            F.when(F.col("product_name").isNotNull(), 1).otherwise(0)
            + F.when(F.col("manufacturer") != "Unknown", 1).otherwise(0)
            + F.when(F.col("category") != "Unknown", 1).otherwise(0)
            + F.when(F.col("price").isNotNull(), 1).otherwise(0)
            + F.when(F.col("has_dimensions"), 1).otherwise(0)
            + F.when(F.col("has_description"), 1).otherwise(0)
            + F.when(F.col("has_about_product"), 1).otherwise(0)


        ).cast("integer"),

    )
)

save_gold(
    amazon_product_catalog,
    "amazon_product_catalog",
)

In [0]:
amazon_products_for_rag = read_silver("amazon_products_for_rag")

amazon_product_documents = (
    amazon_products_for_rag
    .withColumn(
        "document_id",
        F.concat(
            F.lit("amazon_product-"),
            F.col("product_id"),
        ),
    )
    .withColumn(
        "document_type",
        F.lit("product_catalog"),
    )
    .withColumn(
        "document_title",
        F.col("product_name")
    )
    .withColumn(
        "metadata",
        F.to_json(
            F.struct(
                F.col("product_id"),
                F.col("manufacturer"),
                F.col("price"),
                F.col("category"),
                F.col("dimensions"),
                F.col("data_completeness_score"),
                F.lit("amazon_dataset").alias("source")
            )
        )
    )
    .select(
        "document_id",
        "document_type",
        "document_title",
        "product_id",
        F.col("searchable_text").alias("document_text"),
        "manufacturer",
        "category",
        "price",
        "dimensions",
        "data_completeness_score",
        "metadata",

    )
    .filter(
        (F.col("data_completeness_score") >= 3)
        & (F.length("document_text") >= 100)
    )
    
)

save_gold(
    amazon_product_documents,
    "amazon_product_documents"
)


In [0]:
amazon_product_quality = (
    amazon_product_catalog
    .groupBy("data_completeness_score")
    .agg(
        F.count("*").alias("product_count"),
        F.round(
            F.avg("price"),
            2,
        ).alias("average_price"),
        F.sum(
            F.when(
                F.col("has_dimensions"),
                1,
            ).otherwise(0)
        ).alias("products_with_dimensions"),
        F.sum(
            F.when(
                F.col("has_description"),
                1,
            ).otherwise(0)
        ).alias("products_with_description"),
    )
    .orderBy("data_completeness_score")
)
save_gold(
    amazon_product_quality,
    "amazon_product_quality",
)